In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================


!pip install -q transformers datasets bitsandbytes accelerate pandas sentencepiece protobuf
print("✅ All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
✅ All dependencies installed.


In [2]:
# ============================================================
# CELL 2 : Configuration
# ============================================================


import torch
import pandas as pd
import json
import re
import gc
from datasets import load_dataset, get_dataset_config_names, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Running on: {DEVICE}")
assert DEVICE == "cuda", "Go to Runtime → Change runtime type → T4 GPU."

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID     = "Qwen/Qwen2.5-3B-Instruct"
QUANTIZATION = "4bit"

# ── Dataset ───────────────────────────────────────────────────────────────────
MSA_DATASET_ID     = "MBZUAI/ArabicMMLU"
DIALECT_DATASET_ID = "QCRI/AraDICE-ArabicMMLU-egy"
MSA_CONFIG         = "All"

# ── Sampling ──────────────────────────────────────────────────────────────────
N_SAMPLES     = 60          # rows per variety → 120 total inferences
VALID_CHOICES = ["A", "B", "C", "D", "E"]

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_CSV = "dialectal_gap_results.csv"

print("✅ Configuration ready.")
print(f"   Model      : {MODEL_ID}  [{QUANTIZATION}]")
print(f"   MSA data   : {MSA_DATASET_ID}")
print(f"   Dialect    : {DIALECT_DATASET_ID}")
print(f"   Samples    : {N_SAMPLES} per variety → {2*N_SAMPLES} total")

🖥️  Running on: cuda
✅ Configuration ready.
   Model      : Qwen/Qwen2.5-3B-Instruct  [4bit]
   MSA data   : MBZUAI/ArabicMMLU
   Dialect    : QCRI/AraDICE-ArabicMMLU-egy
   Samples    : 60 per variety → 120 total


In [3]:
# ============================================================
# CELL 3 : Load & Normalise Both Datasets
# ============================================================

import pandas as pd
import json
from datasets import load_dataset, get_dataset_config_names, concatenate_datasets

# ──  Load MSA dataset ──────────────────────────────────────────────────────
print(f"⏳ Loading MSA dataset: {MSA_DATASET_ID} …")
msa_raw = load_dataset(MSA_DATASET_ID, MSA_CONFIG, split="test")
MAX_ROWS = 6000
msa_raw = msa_raw.shuffle(seed=42).select(range(MAX_ROWS))
print(f"   Raw MSA rows : {len(msa_raw):,}")
print(f"   Columns      : {msa_raw.column_names}")

# Convert directly to Pandas — avoids all Arrow type-cast issues.
df_msa_raw = msa_raw.to_pandas()

def normalise_msa_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise the ArabicMMLU Pandas DataFrame into the unified schema.
    All string coercions happen in Pandas, which is null-tolerant.
    """
    out = pd.DataFrame()
    out["text_variety"]   = "MSA"
    out["question"]       = df["Question"].fillna("").astype(str)
    out["option_A"]       = df["Option 1"].fillna("").astype(str)
    out["option_B"]       = df["Option 2"].fillna("").astype(str)
    out["option_C"]       = df["Option 3"].fillna("").astype(str)
    out["option_D"]       = df["Option 4"].fillna("").astype(str)
    # Option 5 is null-typed in Arrow but Pandas reads it as float NaN.
    # Convert to empty string uniformly.
    out["option_E"]       = df["Option 5"].fillna("").astype(str).str.strip()
    out["option_E"]       = out["option_E"].replace({"nan": "", "None": ""})
    out["answer_key"]     = df["Answer Key"].fillna("").astype(str).str.strip().str.upper()
    out["subject"]        = df["Subject"].fillna("").astype(str)
    out["level"]          = df["Level"].fillna("").astype(str)
    out["source_dataset"] = MSA_DATASET_ID
    return out

df_msa_normalised = normalise_msa_df(df_msa_raw)
print(f"   Normalised MSA rows: {len(df_msa_normalised):,}")

# ── Load Dialect dataset (all 40 subject configs) ─────────────────────────
print(f"\n⏳ Loading Dialect dataset: {DIALECT_DATASET_ID} …")
dialect_configs = get_dataset_config_names(DIALECT_DATASET_ID)
print(f"   Found {len(dialect_configs)} subject configs — concatenating …")
MAX_ROWS_PER_CONFIG = 150
dialect_splits = []
for cfg in dialect_configs:
    try:
        ds = load_dataset(DIALECT_DATASET_ID, cfg, split="test")
        ds = ds.shuffle(seed=42).select(range(min(MAX_ROWS_PER_CONFIG, len(ds))))

        dialect_splits.append(ds.to_pandas())
    except Exception as e:
        print(f"   ⚠️  Skipping config '{cfg}': {e}")

df_dialect_raw = pd.concat(dialect_splits, ignore_index=True)
print(f"   Raw Dialect rows : {len(df_dialect_raw):,}")
print(f"   Columns          : {df_dialect_raw.columns.tolist()}")

def parse_options(opts_raw) -> dict:
    """
    Safely parse the 'options' column, which may arrive as a Python
    dict (already parsed by Pandas) or as a JSON string.
    Returns a dict with keys A–E (values may be None).
    """
    if isinstance(opts_raw, dict):
        return opts_raw
    if isinstance(opts_raw, str):
        try:
            return json.loads(opts_raw)
        except Exception:
            return {}
    return {}

def normalise_dialect_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise the AraDiCE-EGY Pandas DataFrame into the unified schema.
    The 'options' column is a dict; we unpack it into flat columns.
    """
    opts = df["options"].apply(parse_options)

    out = pd.DataFrame()
    out["text_variety"]   = "EGY_DIALECT"
    out["question"]       = df["question"].fillna("").astype(str)
    out["option_A"]       = opts.apply(lambda o: str(o.get("A") or "")).str.strip()
    out["option_B"]       = opts.apply(lambda o: str(o.get("B") or "")).str.strip()
    out["option_C"]       = opts.apply(lambda o: str(o.get("C") or "")).str.strip()
    out["option_D"]       = opts.apply(lambda o: str(o.get("D") or "")).str.strip()
    # Option E is always null in EGY but we keep the column for schema parity.
    out["option_E"]       = opts.apply(lambda o: str(o.get("E") or "")).str.strip()
    out["option_E"]       = out["option_E"].replace({"None": "", "nan": ""})
    out["answer_key"]     = df["Answer Key"].fillna("").astype(str).str.strip().str.upper()
    out["subject"]        = df["Subject"].fillna("").astype(str)
    out["level"]          = df["Level"].fillna("").astype(str)
    out["source_dataset"] = DIALECT_DATASET_ID
    return out

df_dialect_normalised = normalise_dialect_df(df_dialect_raw)
print(f"   Normalised Dialect rows: {len(df_dialect_normalised):,}")

# ── Filter and sample ─────────────────────────────────────────────────────
def is_valid_row(df: pd.DataFrame) -> pd.Series:
    """
    Boolean mask: keep rows that have a non-empty question, a valid
    answer key letter, and at least option A filled in.
    """
    return (
        df["question"].str.strip().astype(bool) &
        df["answer_key"].isin(VALID_CHOICES) &
        df["option_A"].str.strip().astype(bool)
    )

df_msa_clean     = df_msa_normalised[is_valid_row(df_msa_normalised)].copy()
df_dialect_clean = df_dialect_normalised[is_valid_row(df_dialect_normalised)].copy()

print(f"\n   Valid MSA rows     : {len(df_msa_clean):,}")
print(f"   Valid Dialect rows : {len(df_dialect_clean):,}")

# Reproducible shuffle + sample.
df_msa     = df_msa_clean.sample(n=min(N_SAMPLES, len(df_msa_clean)),
                                  random_state=42).reset_index(drop=True)
df_dialect = df_dialect_clean.sample(n=min(N_SAMPLES, len(df_dialect_clean)),
                                      random_state=42).reset_index(drop=True)

print(f"\n✅ Dataset ready.")
print(f"   MSA sample     : {len(df_msa)} rows")
print(f"   Dialect sample : {len(df_dialect)} rows")

print(f"\nMSA sample — first 2 rows:")
print(df_msa[["question","option_A","option_B","answer_key","subject"]].head(2).to_string())
print(f"\nDialect sample — first 2 rows:")
print(df_dialect[["question","option_A","option_B","answer_key","subject"]].head(2).to_string())

⏳ Loading MSA dataset: MBZUAI/ArabicMMLU …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

dev.csv: 0.00B [00:00, ?B/s]

...

test.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/210 [00:00<?, ? examples/s]

test.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/64 [00:00<?, ? examples/s]

   Raw Dialect rows : 5,126
   Columns          : ['ID', 'Group', 'is_few_shot', 'Level', 'Country', 'Source', 'Subject', 'level_group_subject', 'question', 'options', 'Answer Key', 'context']
   Normalised Dialect rows: 5,126

   Valid MSA rows     : 6,000
   Valid Dialect rows : 5,126

✅ Dataset ready.
   MSA sample     : 60 rows
   Dialect sample : 60 rows

MSA sample — first 2 rows:
                                                     question                option_A                  option_B answer_key          subject
0  يحظر ترك او القاء او وضع اى شي من شانة ان يعوق حركة المرور                     خطأ                      صحيح          B     Driving Test
1               كان عذاب الله تعالى للذين كفروا من قوم نوح :   أرسل عليهم ريحاً صرصر   أرسل عليهم طيراً أبابيل           D  Islamic Studies

Dialect sample — first 2 rows:
                                                                                                                  question                      option_A      

In [4]:
# ============================================================
# CELL 4: Free Memory Before Model Load
# ============================================================

import gc, torch

# Delete the large intermediate HF dataset objects that are
# no longer needed (we already have df_msa and df_dialect).
for _var in ["msa_raw", "df_msa_raw", "df_msa_normalised", "df_msa_clean",
             "df_dialect_raw", "df_dialect_normalised", "df_dialect_clean",
             "dialect_splits"]:
    if _var in dir():
        del globals()[_var]

# Python garbage collector — frees reference-counted objects.
gc.collect()

# PyTorch CUDA allocator — returns cached-but-free blocks to the OS.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
free      = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_reserved()) / 1e9

print("Memory freed.")
print(f"   Allocated : {allocated:.2f} GB")
print(f"   Reserved  : {reserved:.2f} GB")
print(f"   Free      : {free:.2f} GB")

# Sanity check — warn if there is not enough headroom for 3B @ 4-bit.
if free < 3.0:
    print("\nLess than 3 GB free. Consider:")
    print("   1. Runtime → Restart runtime, then re-run Cells 1–4.5 in order.")
    print("   2. Or upgrade to Colab Pro for more VRAM.")
else:
    print("\nSufficient VRAM available. Proceed to Cell 5 (model load).")

Memory freed.
   Allocated : 0.00 GB
   Reserved  : 0.00 GB
   Free      : 15.64 GB

Sufficient VRAM available. Proceed to Cell 5 (model load).


In [5]:
# ============================================================
# CELL 5: Load Model & Tokenizer
# ============================================================

def build_bnb_config(mode: str) -> BitsAndBytesConfig:
    if mode == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    return BitsAndBytesConfig(load_in_8bit=True)

print(f"Loading tokenizer for '{MODEL_ID}' …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model [{QUANTIZATION} quantization] — takes ~2–4 min …")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=build_bnb_config(QUANTIZATION),
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print(f"Model loaded.")
print(f"   VRAM allocated : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading tokenizer for 'Qwen/Qwen2.5-3B-Instruct' …


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
   VRAM allocated : 2.07 GB


In [6]:
# ============================================================
# CELL 6: Prompt Engineering for MCQ
# ============================================================
# The prompt presents all available options (skipping blank ones)
# and asks for a single letter.  This is the standard zero-shot
# MCQ format used in the ArabicMMLU and AraDiCE papers themselves,
# making our results directly comparable to published baselines.
#
# Design decisions:
#   1. English system prompt — more robust for multilingual
#      instruction-tuned models (validated by ArabicMMLU authors).
#   2. Arabic question + options in the user turn — preserves the
#      original script and any dialectal morphology.
#   3. "Respond with ONLY the letter" — enforces a 1-token answer.
# ============================================================

SYSTEM_PROMPT = (
    "You are an expert Arabic-language exam assistant. "
    "You will be given a multiple-choice question in Arabic. "
    "Read the question carefully and select the single best answer. "
    "Respond with ONLY the letter of the correct answer (A, B, C, D, or E). "
    "Do NOT write the answer text, any explanation, or any punctuation."
)

def build_mcq_prompt(row: pd.Series) -> list[dict]:
    """
    Build a chat-formatted message list for a single MCQ row.

    Constructs the options block dynamically — options that are
    empty strings (or the string 'nan') are silently omitted.
    This handles both 4-option (AraDiCE-EGY) and 5-option
    (ArabicMMLU) questions with the same function.

    Parameters
    ----------
    row : pd.Series
        A single row from df_msa or df_dialect.

    Returns
    -------
    list[dict]  Chat messages in {"role", "content"} format.
    """
    # Build the options block, skipping empty / null options.
    option_labels = ["A", "B", "C", "D", "E"]
    option_cols   = ["option_A", "option_B", "option_C", "option_D", "option_E"]

    options_text = ""
    for label, col in zip(option_labels, option_cols):
        val = str(row.get(col, "")).strip()
        # Skip empty, 'nan', or 'None' strings.
        if val and val.lower() not in ("nan", "none", ""):
            options_text += f"  {label}) {val}\n"

    user_content = (
        f"السؤال:\n{row['question']}\n\n"   # "Question:" in Arabic
        f"الخيارات:\n{options_text}\n"        # "Options:" in Arabic
        f"الجواب (حرف واحد فقط):"             # "Answer (one letter only):"
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


# ── Smoke-test the prompt renderer ───────────────────────────────────────────
sample_row = df_msa.iloc[0]
rendered = tokenizer.apply_chat_template(
    build_mcq_prompt(sample_row),
    tokenize=False,
    add_generation_prompt=True,
)
print("── Sample rendered prompt ────────────────────────────────")
print(rendered)
print(f"\n   Correct answer: {sample_row['answer_key']}")
print("──────────────────────────────────────────────────────────")

── Sample rendered prompt ────────────────────────────────
<|im_start|>system
You are an expert Arabic-language exam assistant. You will be given a multiple-choice question in Arabic. Read the question carefully and select the single best answer. Respond with ONLY the letter of the correct answer (A, B, C, D, or E). Do NOT write the answer text, any explanation, or any punctuation.<|im_end|>
<|im_start|>user
السؤال:
يحظر ترك او القاء او وضع اى شي من شانة ان يعوق حركة المرور

الخيارات:
  A) خطأ
  B) صحيح

الجواب (حرف واحد فقط):<|im_end|>
<|im_start|>assistant


   Correct answer: B
──────────────────────────────────────────────────────────


In [7]:
# ============================================================
# CELL 7: Inference Helper
# ============================================================

def run_mcq_inference(row: pd.Series) -> tuple[str, str]:
    """
    Run zero-shot MCQ inference on a single question row.

    Strategy:
      • Generate up to 5 new tokens (a letter + possible punctuation).
      • Parse the first valid A–E letter found in the output.
      • Return 'UNKNOWN' if no valid letter is found.

    Returns
    -------
    raw_output      : str   Full decoded model output (for debugging).
    predicted_label : str   One of {'A','B','C','D','E','UNKNOWN'}.
    """
    messages   = build_mcq_prompt(row)
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_str,
        return_tensors="pt",
        truncation=True,
        max_length=1024,   # questions can be long; 1024 is safe
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=5,       # letter + optional punctuation
            do_sample=False,        # greedy — deterministic for research
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # ── Parse: find first valid letter in the output ──────────────────────
    # Upper-case, strip whitespace and common punctuation first.
    cleaned = raw_output.upper().strip(" .,\n\t'\"():؟")

    # Regex: first occurrence of a standalone A–E, with or without
    # surrounding parentheses like "(A)" or "A."
    match = re.search(r"\b([A-E])\b", cleaned)
    if match:
        predicted = match.group(1)
    elif cleaned and cleaned[0] in VALID_CHOICES:
        # Fallback: take the very first character if it's a valid letter.
        predicted = cleaned[0]
    else:
        predicted = "UNKNOWN"

    return raw_output, predicted

In [8]:
# ============================================================
# CELL 8: Full Evaluation Loop
# ============================================================
# Evaluates all MSA rows first, then all dialect rows.
# Progress is printed every 10 rows with a running accuracy
# for each variety separately — so you can watch the gap form
# in real time.
#
# Expected runtime on Colab T4 (60+60=120 rows):
#   Qwen2.5-7B @ 4-bit → ~3–5 s/row → ~6–10 min total
# ============================================================

def evaluate_dataframe(df: pd.DataFrame, variety_label: str) -> list[dict]:
    """
    Run inference over every row in `df` and collect results.

    Parameters
    ----------
    df            : pd.DataFrame  Normalised question rows.
    variety_label : str           'MSA' or 'EGY_DIALECT'.

    Returns
    -------
    list[dict]  One result dict per row.
    """
    results  = []
    n_correct = 0

    print(f"\n{'='*60}")
    print(f"  Evaluating: {variety_label}  ({len(df)} questions)")
    print(f"{'='*60}")

    for i, (_, row) in enumerate(df.iterrows()):
        raw_output, predicted = run_mcq_inference(row)
        true_answer = row["answer_key"]
        is_correct  = (predicted == true_answer)
        if is_correct:
            n_correct += 1

        results.append({
            "variety"         : variety_label,
            "question"        : row["question"],
            "option_A"        : row["option_A"],
            "option_B"        : row["option_B"],
            "option_C"        : row["option_C"],
            "option_D"        : row["option_D"],
            "true_answer"     : true_answer,
            "predicted_answer": predicted,
            "raw_model_output": raw_output,
            "is_correct"      : is_correct,
            "subject"         : row["subject"],
            "level"           : row["level"],
            "source_dataset"  : row["source_dataset"],
        })

        # ── Live progress every 10 rows ──────────────────────────────────
        if (i + 1) % 10 == 0 or (i + 1) == len(df):
            running_acc = n_correct / (i + 1)
            unknowns    = sum(1 for r in results if r["predicted_answer"] == "UNKNOWN")
            print(
                f"  [{i+1:>3}/{len(df)}]  "
                f"Running Acc = {running_acc:.2%}  |  "
                f"UNKNOWN outputs = {unknowns}"
            )

    return results

# Run both varieties sequentially.
results_msa     = evaluate_dataframe(df_msa,     variety_label="MSA")
results_dialect = evaluate_dataframe(df_dialect, variety_label="EGY_DIALECT")

# Combine into a single DataFrame.
all_results = pd.DataFrame(results_msa + results_dialect)
print(f"\nInference complete.  Total rows: {len(all_results)}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



  Evaluating: MSA  (60 questions)
  [ 10/60]  Running Acc = 60.00%  |  UNKNOWN outputs = 0
  [ 20/60]  Running Acc = 70.00%  |  UNKNOWN outputs = 0
  [ 30/60]  Running Acc = 66.67%  |  UNKNOWN outputs = 0
  [ 40/60]  Running Acc = 70.00%  |  UNKNOWN outputs = 0
  [ 50/60]  Running Acc = 66.00%  |  UNKNOWN outputs = 0
  [ 60/60]  Running Acc = 61.67%  |  UNKNOWN outputs = 0

  Evaluating: EGY_DIALECT  (60 questions)
  [ 10/60]  Running Acc = 60.00%  |  UNKNOWN outputs = 0
  [ 20/60]  Running Acc = 40.00%  |  UNKNOWN outputs = 0
  [ 30/60]  Running Acc = 33.33%  |  UNKNOWN outputs = 0
  [ 40/60]  Running Acc = 30.00%  |  UNKNOWN outputs = 0
  [ 50/60]  Running Acc = 32.00%  |  UNKNOWN outputs = 0
  [ 60/60]  Running Acc = 28.33%  |  UNKNOWN outputs = 0

Inference complete.  Total rows: 120


In [9]:
# ============================================================
# CELL 9: Metrics — The Dialectal Gap
# ============================================================
# This cell produces the three key numbers your paper needs:
#   1. MSA accuracy
#   2. Dialect accuracy
#   3. The gap  (MSA acc − Dialect acc)
#
# It also produces a subject-level breakdown — essential for
# the discussion section: some subjects (e.g. History) may
# show a larger gap than others (e.g. Math), revealing where
# dialect interferes most with reasoning.
# ============================================================

print(f"\n{'='*60}")
print(f"  MODEL : {MODEL_ID}  [{QUANTIZATION}]")
print(f"  TASK  : Zero-Shot MCQ  (ArabicMMLU vs. AraDiCE-EGY)")
print(f"{'='*60}")

# ── Overall accuracy per variety ──────────────────────────────────────────────
overall = (
    all_results
    .groupby("variety")["is_correct"]
    .agg(n_correct="sum", n_total="count")
    .assign(accuracy=lambda x: x["n_correct"] / x["n_total"])
    .round(4)
)
print("\nOverall Accuracy by Variety:")
print(overall.to_string())

msa_acc     = overall.loc["MSA",         "accuracy"]
dialect_acc = overall.loc["EGY_DIALECT", "accuracy"]
gap         = msa_acc - dialect_acc

print(f"\n  MSA accuracy     : {msa_acc:.2%}")
print(f"  Dialect accuracy : {dialect_acc:.2%}")
print(f"  ── Dialectal Gap : {gap:+.2%}  ({'MSA better' if gap > 0 else 'Dialect better'})")

# ── UNKNOWN / parse-failure rate ─────────────────────────────────────────────
unknown_rate = (all_results["predicted_answer"] == "UNKNOWN").mean()
print(f"\n  Parse-failure rate (UNKNOWN): {unknown_rate:.2%}")

# ── Per-subject accuracy breakdown ────────────────────────────────────────────
print("\nPer-Subject Accuracy (subjects with ≥ 5 rows in each variety):")
subject_stats = (
    all_results
    .groupby(["subject", "variety"])["is_correct"]
    .agg(n_correct="sum", n_total="count")
    .assign(accuracy=lambda x: (x["n_correct"] / x["n_total"]).round(3))
    .reset_index()
    .pivot(index="subject", columns="variety", values="accuracy")
    .dropna()                          # keep subjects present in both varieties
    .assign(gap=lambda x: x.get("MSA", 0) - x.get("EGY_DIALECT", 0))
    .sort_values("gap", ascending=False)
)
print(subject_stats.to_string())

# ── Answer distribution (catch systematic bias) ───────────────────────────────
print("\nPredicted Answer Distribution (should be spread across A–E):")
dist = (
    all_results
    .groupby(["variety", "predicted_answer"])
    .size()
    .unstack(fill_value=0)
)
print(dist.to_string())


  MODEL : Qwen/Qwen2.5-3B-Instruct  [4bit]
  TASK  : Zero-Shot MCQ  (ArabicMMLU vs. AraDiCE-EGY)

Overall Accuracy by Variety:
             n_correct  n_total  accuracy
variety                                  
EGY_DIALECT         17       60    0.2833
MSA                 37       60    0.6167

  MSA accuracy     : 61.67%
  Dialect accuracy : 28.33%
  ── Dialectal Gap : +33.34%  (MSA better)

  Parse-failure rate (UNKNOWN): 0.00%

Per-Subject Accuracy (subjects with ≥ 5 rows in each variety):
variety                    EGY_DIALECT    MSA    gap
subject                                             
Arabic Language (General)        0.000  1.000  1.000
History                          0.143  1.000  0.857
Arabic Language (Grammar)        0.000  0.800  0.800
Social Science                   0.000  0.667  0.667
Driving Test                     0.000  0.615  0.615
Islamic Studies                  0.333  0.875  0.542
Geography                        0.000  0.500  0.500
Biology                 

In [10]:
# ============================================================
# DIAGNOSTIC CELL B: Interactive Algorithm Walkthrough
# ============================================================
# Tests the FULL pipeline (prompt build → tokenise → generate
# → parse → evaluate) on manually typed Arabic questions.
# Run this AFTER Cell 5 (model loaded) and Cell 6 (prompt fn).
#
# No real dataset needed — you provide the question yourself.
# Perfect for verifying the pipeline logic before the 120-row run.
# ============================================================

# ── B0. Helper: pretty-print a full pipeline trace ────────────
def run_interactive_example(
    question   : str,
    option_a   : str,
    option_b   : str,
    option_c   : str,
    option_d   : str,
    true_answer: str,          # "A", "B", "C", or "D"
    option_e   : str = "",     # optional 5th option
    label      : str = "MANUAL_TEST",
):
    """
    Runs one question through the complete inference pipeline and
    prints a step-by-step trace so you can verify every stage:
      1. Prompt construction
      2. Tokenisation (token count)
      3. Raw model output
      4. Parsing logic
      5. Correctness verdict
    """
    print("\n" + "═" * 62)
    print(f"  TEST: {label}")
    print("═" * 62)

    # Build a mock DataFrame row (same schema as df_msa / df_dialect)
    row = pd.Series({
        "question"      : question,
        "option_A"      : option_a,
        "option_B"      : option_b,
        "option_C"      : option_c,
        "option_D"      : option_d,
        "option_E"      : option_e,
        "answer_key"    : true_answer.strip().upper(),
        "subject"       : "Manual",
        "level"         : "N/A",
        "source_dataset": "user_input",
        "text_variety"  : label,
    })

    # ── Step 1: Build and display the prompt ──────────────────
    messages   = build_mcq_prompt(row)
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print("\n📋 STEP 1 — Rendered Prompt:")
    print("─" * 40)
    print(prompt_str)
    print("─" * 40)

    # ── Step 2: Tokenise and show token count ─────────────────
    inputs     = tokenizer(
        prompt_str, return_tensors="pt",
        truncation=True, max_length=1024
    ).to(DEVICE)
    n_tokens   = inputs["input_ids"].shape[-1]
    print(f"\n🔢 STEP 2 — Tokenisation:")
    print(f"   Input token count : {n_tokens}")

    # ── Step 3: Generate ──────────────────────────────────────
    print(f"\n⚙️  STEP 3 — Generating (max_new_tokens=5, greedy) …")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, n_tokens:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f"   Raw model output  : '{raw_output}'")
    print(f"   New tokens emitted: {len(new_tokens)}")

    # ── Step 4: Parse the answer letter ───────────────────────
    cleaned = raw_output.upper().strip(" .,\n\t'\"():؟")
    match   = re.search(r"\b([A-E])\b", cleaned)
    if match:
        predicted = match.group(1)
        parse_method = "regex word-boundary match"
    elif cleaned and cleaned[0] in VALID_CHOICES:
        predicted    = cleaned[0]
        parse_method = "first-character fallback"
    else:
        predicted    = "UNKNOWN"
        parse_method = "no valid letter found"

    print(f"\n🔍 STEP 4 — Parsing:")
    print(f"   Cleaned string : '{cleaned}'")
    print(f"   Parse method   : {parse_method}")
    print(f"   Predicted      : '{predicted}'")

    # ── Step 5: Evaluate ──────────────────────────────────────
    true_ans   = true_answer.strip().upper()
    is_correct = (predicted == true_ans)
    verdict    = "✅  CORRECT" if is_correct else "❌  WRONG"

    print(f"\n🏆 STEP 5 — Verdict:")
    print(f"   True answer : '{true_ans}'")
    print(f"   Predicted   : '{predicted}'")
    print(f"   Result      : {verdict}")
    print("═" * 62)

    return {
        "label"          : label,
        "question"       : question,
        "true_answer"    : true_ans,
        "predicted"      : predicted,
        "raw_output"     : raw_output,
        "is_correct"     : is_correct,
        "parse_method"   : parse_method,
        "input_tokens"   : n_tokens,
    }


# ════════════════════════════════════════════════════════════
# ── EXAMPLE 1: MSA — General Knowledge (easy, clear MSA)
# ════════════════════════════════════════════════════════════
result_1 = run_interactive_example(
    question    = "ما هي عاصمة المملكة العربية السعودية؟",
    option_a    = "جدة",
    option_b    = "الرياض",
    option_c    = "مكة المكرمة",
    option_d    = "الدمام",
    true_answer = "B",
    label       = "MSA — Geography (Easy)",
)

# ════════════════════════════════════════════════════════════
# ── EXAMPLE 2: Egyptian Dialect — Same question rewritten
#    in Egyptian Arabic (Levantine/colloquial phrasing).
#    Expected: model may struggle more here.
# ════════════════════════════════════════════════════════════
result_2 = run_interactive_example(
    question    = "إيه هي عاصمة المملكة العربية السعودية؟",   # إيه = EGY for ما
    option_a    = "جدة",
    option_b    = "الرياض",
    option_c    = "مكة المكرمة",
    option_d    = "الدمام",
    true_answer = "B",
    label       = "EGY DIALECT — Same Question, Dialectal Phrasing",
)

# ════════════════════════════════════════════════════════════
# ── EXAMPLE 3: MSA — History (medium difficulty)
# ════════════════════════════════════════════════════════════
result_3 = run_interactive_example(
    question    = "في أي عام تأسست جامعة الدول العربية؟",
    option_a    = "١٩٤٥",
    option_b    = "١٩٤٨",
    option_c    = "١٩٥٢",
    option_d    = "١٩٣٢",
    true_answer = "A",
    label       = "MSA — History (Medium)",
)

# ════════════════════════════════════════════════════════════
# ── EXAMPLE 4: Egyptian Dialect — Science question
#    Uses dialectal vocabulary: بتتكوّن (formed), إيه (what)
# ════════════════════════════════════════════════════════════
result_4 = run_interactive_example(
    question    = "المجموعة الشمسية بتتكوّن من إيه؟",
    option_a    = "الشمس والكواكب والأقمار",
    option_b    = "الشمس بس",
    option_c    = "النجوم والمجرات",
    option_d    = "الأرض والقمر",
    true_answer = "A",
    label       = "EGY DIALECT — Science (Medium)",
)

# ════════════════════════════════════════════════════════════
# ── EXAMPLE 5: ADVERSARIAL — Deliberately ambiguous output
#    Tests the parser's fallback behaviour.
#    We monkey-patch one call to simulate a noisy model output.
# ════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("  ADVERSARIAL TEST — Parser stress test (no model call)")
print("═" * 62)

adversarial_outputs = [
    ("A",           "clean single letter"),
    ("(B)",         "letter in parentheses"),
    ("C.",          "letter with full stop"),
    ("  d  ",       "lowercase with spaces"),
    ("الإجابة: A",  "Arabic prefix before letter"),
    ("Option B is correct", "verbose English answer"),
    ("Yes",         "invalid — no letter → UNKNOWN"),
    ("",            "empty string → UNKNOWN"),
    ("٣",           "Arabic numeral → UNKNOWN"),
]

print(f"\n{'Raw Output':<30} {'Cleaned':<25} {'Parsed':<10} {'Method'}")
print("─" * 80)
for raw, description in adversarial_outputs:
    cleaned = raw.upper().strip(" .,\n\t'\"():؟")
    match   = re.search(r"\b([A-E])\b", cleaned)
    if match:
        parsed = match.group(1);  method = "regex"
    elif cleaned and cleaned[0] in VALID_CHOICES:
        parsed = cleaned[0];      method = "first-char"
    else:
        parsed = "UNKNOWN";       method = "fallback"
    print(f"'{raw:<28}' → '{cleaned:<23}' → {parsed:<10} ({method}) | {description}")

# ════════════════════════════════════════════════════════════
# ── MINI REPORT: summary of the 4 live model tests
# ════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("  MINI REPORT — Interactive Test Results")
print("═" * 62)
results = [result_1, result_2, result_3, result_4]
for r in results:
    tick = "✅" if r["is_correct"] else "❌"
    print(f"  {tick}  [{r['label']}]")
    print(f"       True={r['true_answer']}  Pred={r['predicted']}  "
          f"Raw='{r['raw_output']}'  Tokens={r['input_tokens']}")

n_correct = sum(r["is_correct"] for r in results)
print(f"\n  Mini accuracy: {n_correct}/{len(results)} = {n_correct/len(results):.0%}")
print("═" * 62)


══════════════════════════════════════════════════════════════
  TEST: MSA — Geography (Easy)
══════════════════════════════════════════════════════════════

📋 STEP 1 — Rendered Prompt:
────────────────────────────────────────
<|im_start|>system
You are an expert Arabic-language exam assistant. You will be given a multiple-choice question in Arabic. Read the question carefully and select the single best answer. Respond with ONLY the letter of the correct answer (A, B, C, D, or E). Do NOT write the answer text, any explanation, or any punctuation.<|im_end|>
<|im_start|>user
السؤال:
ما هي عاصمة المملكة العربية السعودية؟

الخيارات:
  A) جدة
  B) الرياض
  C) مكة المكرمة
  D) الدمام

الجواب (حرف واحد فقط):<|im_end|>
<|im_start|>assistant

────────────────────────────────────────

🔢 STEP 2 — Tokenisation:
   Input token count : 129

⚙️  STEP 3 — Generating (max_new_tokens=5, greedy) …
   Raw model output  : 'B'
   New tokens emitted: 2

🔍 STEP 4 — Parsing:
   Cleaned string : 'B'
   Parse m

In [11]:
# ============================================================
# CELL 10: Export Results to CSV
# ============================================================
# The CSV is structured for three uses:
#   1. Spot-check: human reviewers can read question + options
#      alongside the model's prediction.
#   2. Statistical tests: load into R or SciPy for McNemar's
#      test on matched question pairs (future extension).
#   3. Paper appendix: direct supplementary material.
# ============================================================

export_cols = [
    "variety",
    "subject",
    "level",
    "true_answer",
    "predicted_answer",
    "is_correct",
    "raw_model_output",
    "question",
    "option_A", "option_B", "option_C", "option_D",
    "source_dataset",
]

all_results[export_cols].to_csv(
    OUTPUT_CSV, index=False, encoding="utf-8-sig"
    # utf-8-sig adds BOM so Excel/LibreOffice renders Arabic correctly.
)

print(f"✅ Results saved → '{OUTPUT_CSV}'")
print(f"   Shape : {all_results[export_cols].shape}")

# Summary statistics at the bottom of the CSV as comments is not
# standard — instead print a machine-readable JSON block that can
# be copy-pasted directly into your paper's results table.
import json as _json
summary = {
    "model"          : MODEL_ID,
    "quantization"   : QUANTIZATION,
    "msa_dataset"    : MSA_DATASET_ID,
    "dialect_dataset": DIALECT_DATASET_ID,
    "n_samples_each" : N_SAMPLES,
    "msa_accuracy"   : round(float(msa_acc),   4),
    "dialect_accuracy": round(float(dialect_acc), 4),
    "dialectal_gap"  : round(float(gap),        4),
    "unknown_rate"   : round(float(unknown_rate), 4),
}
print("\n📋 Paper-ready summary (JSON):")
print(_json.dumps(summary, indent=2))

# ── Download in Colab ─────────────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print("\n📥 Download triggered.")
except ImportError:
    print("\nℹ️  Find the CSV in your working directory.")

✅ Results saved → 'dialectal_gap_results.csv'
   Shape : (120, 13)

📋 Paper-ready summary (JSON):
{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "quantization": "4bit",
  "msa_dataset": "MBZUAI/ArabicMMLU",
  "dialect_dataset": "QCRI/AraDICE-ArabicMMLU-egy",
  "n_samples_each": 60,
  "msa_accuracy": 0.6167,
  "dialect_accuracy": 0.2833,
  "dialectal_gap": 0.3334,
  "unknown_rate": 0.0
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Download triggered.
